# Original Nutrition MLP Training — Source Extract

This notebook copies original experiment cells **112–114**, including Nutrition5k metadata and image retrieval, extraction of 768-dimensional DINOv2 features, Ridge baseline, six-output MLP training, checkpoint selection, and held-out evaluation. It requires the team's DINOv2 checkpoint (`MyDrive/NutriVision_AI/dinov2_last4_best.pth`) and external Nutrition5k assets. The source asserts the historical 3,262-image shape and 507 test images, so changes in the upstream dataset may require additional investigation. **Source provenance only: the notebook has not been rerun during this packaging step.**


In [ ]:
%pip -q install timm tqdm numpy pandas scikit-learn joblib requests huggingface_hub
from google.colab import drive
drive.mount('/content/drive')


## Download and preprocess Nutrition5k; extract features; Ridge baseline (source cell 112)

In [ ]:

import requests
import numpy as np
import pandas as pd
import torch
import timm
import joblib

from pathlib import Path
from PIL import Image
from huggingface_hub import snapshot_download
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm

root = Path("/content/drive/MyDrive/NutriVision_AI")
data_dir = Path("/content/data/nutrition5k")
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Download RGB images and metadata
snapshot_download(
    repo_id="ai5labsOfficial/nutrition5k",
    repo_type="dataset",
    allow_patterns=["images/*.png", "metadata.csv"],
    local_dir=str(data_dir),
    max_workers=8
)

df = pd.read_csv(data_dir / "metadata.csv")
df["dish_id"] = df["dish_id"].astype(str)

# 2. Load official train and test IDs
base_url = (
    "https://storage.googleapis.com/"
    "nutrition5k_dataset/nutrition5k_dataset/"
    "dish_ids/splits/"
)

def get_ids(filename):
    response = requests.get(base_url + filename, timeout=30)
    response.raise_for_status()
    return set(response.text.split())

official_train = get_ids("depth_train_ids.txt")
official_test = get_ids("depth_test_ids.txt")

available = {
    p.stem for p in (data_dir / "images").glob("*.png")
}

train_ids = sorted(
    set(df.dish_id) & available & official_train
)

test_ids = sorted(
    set(df.dish_id) & available & official_test
)

assert len(test_ids) == 507
assert set(train_ids).isdisjoint(test_ids)

print("Official train images:", len(train_ids))
print("Official test images:", len(test_ids))

# 3. Prepare nutrition targets
targets = [
    "total_calories",
    "total_mass_g",
    "fat_g",
    "carb_g",
    "protein_g"
]

df = df.set_index("dish_id")

# Use same visual preprocessing as our DINOv2
transform = T.Compose([
    T.Resize(320, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(280),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

class NutritionImages(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        dish_id = self.ids[idx]
        image = Image.open(
            data_dir / "images" / f"{dish_id}.png"
        ).convert("RGB")

        y = df.loc[dish_id, targets].to_numpy(
            dtype=np.float32
        )

        return transform(image), torch.tensor(y)

# 4. Load OUR trained DINOv2
ckpt = torch.load(
    root / "dinov2_last4_best.pth",
    map_location="cpu",
    weights_only=True
)

dino = timm.create_model(
    "vit_base_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=101,
    img_size=280
)

dino.load_state_dict(ckpt["model_state_dict"])
dino = dino.to(device).eval()

# 5. Extract features only once, without training
all_ids = train_ids + test_ids

loader = DataLoader(
    NutritionImages(all_ids),
    batch_size=16,
    shuffle=False,
    num_workers=0
)

features = []
nutrition_labels = []

with torch.inference_mode():
    for images, y in tqdm(
        loader, desc="Extracting Nutrition Features"
    ):
        images = images.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(device == "cuda")
        ):
            x = dino.forward_features(images)
            x = dino.forward_head(x, pre_logits=True)

        features.append(x.float().cpu())
        nutrition_labels.append(y)

X = torch.cat(features).numpy()
Y = torch.cat(nutrition_labels).numpy()

assert X.ndim == 2
assert X.shape[0] == len(all_ids)
assert np.isfinite(X).all()
assert np.isfinite(Y).all()

# Save small feature dataset permanently in Drive
torch.save({
    "features": torch.from_numpy(X),
    "targets": torch.from_numpy(Y),
    "ids": all_ids,
    "train_ids": train_ids,
    "test_ids": test_ids,
    "target_names": targets
}, root / "nutrition_dinov2_features.pth")

print("Feature shape:", X.shape)

# 6. Train/validation split, excluding official test
train_idx, val_idx = train_test_split(
    np.arange(len(train_ids)),
    test_size=0.15,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx])
X_val = scaler.transform(X[val_idx])

y_train = Y[train_idx]
y_val = Y[val_idx]

# Standardize targets using training values only
y_mean = y_train.mean(axis=0)
y_std = y_train.std(axis=0).clip(min=1e-6)

# 7. Train a lightweight five-output nutrition head
best_mae = float("inf")
best_model = None

print("\n===== NUTRITION HEAD TRAINING =====")

for alpha in [1.0, 10.0, 100.0]:
    regressor = Ridge(alpha=alpha)

    regressor.fit(
        X_train,
        (y_train - y_mean) / y_std
    )

    pred = regressor.predict(X_val) * y_std + y_mean
    pred = np.maximum(pred, 0)

    calorie_mae = mean_absolute_error(
        y_val[:, 0], pred[:, 0]
    )

    print(
        f"Ridge alpha={alpha}: "
        f"Calorie Validation MAE={calorie_mae:.2f} kcal"
    )

    if calorie_mae < best_mae:
        best_mae = calorie_mae
        best_model = regressor
        best_alpha = alpha

# 8. Save trained nutrition head
joblib.dump({
    "regressor": best_model,
    "feature_scaler": scaler,
    "target_mean": y_mean,
    "target_std": y_std,
    "target_names": targets,
    "alpha": best_alpha
}, root / "nutrition_dinov2_head.joblib")

torch.save({
    "train_ids": [train_ids[i] for i in train_idx],
    "val_ids": [train_ids[i] for i in val_idx],
    "test_ids": test_ids
}, root / "nutrition_experiment_split.pth")

print("\n===== FINAL NUTRITION RESULT =====")
print(f"Best Validation MAE: {best_mae:.2f} kcal")
print(f"Best alpha: {best_alpha}")
print("Nutrition head saved.")
print("Official test set untouched in this experiment.")


## Train released Nutrition MLP (source cell 113)

In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

root = Path("/content/drive/MyDrive/NutriVision_AI")

data = torch.load(
    root / "nutrition_dinov2_features.pth",
    map_location="cpu",
    weights_only=True
)

X = data["features"].float().numpy()
Y = data["targets"].float().numpy()

n_train = len(data["train_ids"])

assert X.shape == (3262, 768)
assert Y.shape == (3262, 5)
assert data["ids"][:n_train] == data["train_ids"]

# Recreate exactly the Ridge validation split
train_idx, val_idx = train_test_split(
    np.arange(n_train),
    test_size=0.15,
    random_state=42
)

# Normalize visual features using training data only
x_mean = X[train_idx].mean(axis=0)
x_std = X[train_idx].std(axis=0).clip(min=1e-6)

Xn = (X - x_mean) / x_std

# Targets: calories, mass, fat, carbs, protein
# Also learn energy density = calories / mass
density = Y[:, 0] / np.maximum(Y[:, 1], 1.0)

Z = np.column_stack([
    np.log1p(np.maximum(Y, 0)),
    np.log(np.maximum(density, 0.01))
]).astype("float32")

z_mean = Z[train_idx].mean(axis=0)
z_std = Z[train_idx].std(axis=0).clip(min=0.05)

Zn = (Z - z_mean) / z_std

train_ds = TensorDataset(
    torch.from_numpy(Xn[train_idx].astype("float32")),
    torch.from_numpy(Zn[train_idx])
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True
)

val_x = torch.from_numpy(
    Xn[val_idx].astype("float32")
)

val_calories = torch.from_numpy(
    Y[val_idx, 0].astype("float32")
)

# Small nonlinear nutrition network
class NutritionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)

model = NutritionMLP()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=0.03
)

zm = torch.tensor(z_mean)
zs = torch.tensor(z_std)

# Convert predicted targets to real calorie values
def calorie_predictions(output):
    raw = output * zs + zm

    direct = torch.expm1(
        raw[:, 0].clamp(0, 9)
    ).clamp_min(0)

    mass = torch.expm1(
        raw[:, 1].clamp(0, 9)
    ).clamp_min(0)

    energy_density = torch.exp(
        raw[:, 5].clamp(-4, 3)
    )

    calculated = mass * energy_density

    return {
        "Direct": direct,
        "Mass x Density": calculated,
        "50/50 Blend": 0.5 * direct + 0.5 * calculated
    }

best_mae = float("inf")
best_method = None
bad_epochs = 0
patience = 12

print("===== NUTRITION MLP TRAINING =====")

for epoch in range(1, 81):
    model.train()

    for images_features, targets in train_loader:
        optimizer.zero_grad()

        outputs = model(images_features)

        loss_per_target = F.smooth_l1_loss(
            outputs,
            targets,
            reduction="none"
        )

        # Give calorie and density prediction extra weight
        weights = torch.tensor(
            [3.0, 1.0, 0.5, 0.5, 0.5, 2.0]
        )

        loss = (
            loss_per_target * weights
        ).mean()

        loss.backward()
        optimizer.step()

    model.eval()

    with torch.inference_mode():
        output = model(val_x)
        predictions = calorie_predictions(output)

        scores = {
            name: torch.abs(
                pred - val_calories
            ).mean().item()
            for name, pred in predictions.items()
        }

    method = min(scores, key=scores.get)
    val_mae = scores[method]

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d}: "
            f"Validation MAE={val_mae:.2f} kcal "
            f"({method})"
        )

    if val_mae < best_mae - 0.05:
        best_mae = val_mae
        best_method = method
        best_epoch = epoch
        bad_epochs = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "feature_mean": torch.from_numpy(x_mean),
            "feature_std": torch.from_numpy(x_std),
            "target_mean": torch.from_numpy(z_mean),
            "target_std": torch.from_numpy(z_std),
            "method": method,
            "epoch": epoch,
            "val_mae": val_mae
        }, root / "nutrition_mlp_best.pth")
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("Early stopping.")
        break

print("\n===== FINAL NUTRITION COMPARISON =====")
print("Ridge Validation MAE: 79.69 kcal")
print(f"MLP Validation MAE: {best_mae:.2f} kcal")
print(f"Best prediction method: {best_method}")
print(f"Best epoch: {best_epoch}")
print(f"Improvement: {79.69-best_mae:+.2f} kcal")

if best_mae < 79.69:
    print("MLP improved on Ridge!")
else:
    print("MLP did not improve. Keep Ridge baseline.")



## Evaluate released MLP on held-out test (source cell 114)

In [ ]:

import torch
import torch.nn as nn
from pathlib import Path

root = Path("/content/drive/MyDrive/NutriVision_AI")

# Restore saved visual features and best nutrition head
data = torch.load(
    root / "nutrition_dinov2_features.pth",
    map_location="cpu",
    weights_only=True
)

ckpt = torch.load(
    root / "nutrition_mlp_best.pth",
    map_location="cpu",
    weights_only=True
)

class NutritionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)

model = NutritionMLP()
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Official test data is at the end of our saved features
n_train = len(data["train_ids"])

assert data["ids"][n_train:] == data["test_ids"]
assert len(data["test_ids"]) == 507
assert ckpt["method"] == "Direct"

X_test = data["features"][n_train:].float()
Y_test = data["targets"][n_train:].float()

# Apply the same training normalization
x_mean = ckpt["feature_mean"].float()
x_std = ckpt["feature_std"].float()

X_test = (X_test - x_mean) / x_std

with torch.inference_mode():
    output = model(X_test)

    # Reverse the target normalization
    raw = (
        output * ckpt["target_std"].float()
        + ckpt["target_mean"].float()
    )

    # Best method: Direct Calories
    predicted = torch.expm1(
        raw[:, 0].clamp(0, 9)
    ).clamp_min(0)

actual = Y_test[:, 0]

errors = predicted - actual
absolute_errors = errors.abs()

mae = absolute_errors.mean().item()
rmse = errors.square().mean().sqrt().item()
median_ae = absolute_errors.median().item()
within_50 = (absolute_errors <= 50).float().mean().item() * 100
within_100 = (absolute_errors <= 100).float().mean().item() * 100

print("===== OFFICIAL NUTRITION5K TEST =====")
print("Test images:", len(actual))
print(f"Calorie MAE: {mae:.2f} kcal")
print(f"Calorie RMSE: {rmse:.2f} kcal")
print(f"Median Absolute Error: {median_ae:.2f} kcal")
print(f"Within ±50 kcal: {within_50:.2f}%")
print(f"Within ±100 kcal: {within_100:.2f}%")

print("\n===== PREVIOUS RESNET18 TEST =====")
print("Previous MAE: 82.60 kcal")
print(f"New model MAE: {mae:.2f} kcal")
print(f"Difference: {82.60 - mae:+.2f} kcal")

print("\n===== SELECTED CHECKPOINT =====")
print("Best validation epoch:", ckpt["epoch"])
print("Validation MAE:", round(float(ckpt["val_mae"]), 2))
print("Prediction method:", ckpt["method"])

torch.save({
    "mae": mae,
    "rmse": rmse,
    "median_ae": median_ae,
    "within_50": within_50,
    "within_100": within_100,
    "predictions": predicted,
    "targets": actual,
    "test_ids": data["test_ids"]
}, root / "nutrition_mlp_test_results.pth")

print("\nTest results saved!")
